In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class RBM(nn.Module):
    def __init__(self, num_visible, num_hidden):
        super(RBM, self).__init__()
        self.num_visible = num_visible
        self.num_hidden = num_hidden
        
        self.W = nn.Parameter(torch.randn(num_hidden, num_visible) * 0.01)
        self.a = nn.Parameter(torch.zeros(num_visible))  # Visible biases
        self.b = nn.Parameter(torch.zeros(num_hidden))  # Hidden biases
    
    def forward(self, v):
        h_prob = torch.sigmoid(F.linear(v, self.W, self.b))
        h_sample = torch.bernoulli(h_prob)
        v_prob = torch.sigmoid(F.linear(h_sample, self.W.t(), self.a))
        return v_prob, h_prob

    def energy(self, v):
        term1 = -torch.matmul(v, self.a)
        term2 = -torch.sum(torch.log1p(torch.exp(F.linear(v, self.W, self.b))), dim=1)
        return term1 + term2

    def sample(self, v, num_steps=1):
        for _ in range(num_steps):
            _, h_prob = self.forward(v)
            h_sample = torch.bernoulli(h_prob)
            v_prob = torch.sigmoid(F.linear(h_sample, self.W.t(), self.a))
            v = torch.bernoulli(v_prob)
        return v

    def train_vmc(self, num_samples=1000, lr=0.01, num_epochs=500):
        optimizer = optim.Adam(self.parameters(), lr=lr)
        for epoch in range(num_epochs):
            v = torch.bernoulli(torch.rand(num_samples, self.num_visible))
            E = self.energy(v)
            loss = E.mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if epoch % 50 == 0:
                print(f"Epoch {epoch}: Energy = {loss.item()}")

    def time_evolution(self, H, dt=0.01, num_steps=100):
        """
        Implements time-dependent variational Monte Carlo (t-VMC) using the Dirac-Frenkel principle with Stochastic Reconfiguration.
        """
        for step in range(num_steps):
            v = torch.bernoulli(torch.rand(1000, self.num_visible))  # Sampling states
            E = self.energy(v)
            
            # Computing the natural gradient update using Stochastic Reconfiguration
            S = torch.matmul(v.t(), v) / v.shape[0]  # approximating Covariance matrix
            F_t = torch.matmul(v.t(), E.unsqueeze(1)) / v.shape[0]  # Energy gradient
            W_update = torch.linalg.solve(S + 1e-4 * torch.eye(S.shape[0]), F_t).squeeze()
            
            # Apply update to RBM parameters
            with torch.no_grad():
                self.W -= dt * W_update.view(self.W.shape)
                self.a -= dt * W_update[:self.num_visible]
                self.b -= dt * W_update[self.num_visible:]
            
            if step % 10 == 0:
                print(f"Step {step}: Energy = {E.mean().item()}")




In [ ]:
num_visible = 10  # Number of spins
num_hidden = 20  # Number of hidden units

rbm = RBM(num_visible, num_hidden)
v = torch.bernoulli(torch.rand(100, num_visible))  # Random spin configurations

# Compute energy
energy = rbm.energy(v)

# Train using Variational Monte Carlo
rbm.train_vmc()



In [ ]:
# Perform time evolution for Transverse field ising model
H = torch.zeros((num_visible, num_visible))
for i in range(num_visible):
    H[i, (i + 1) % num_visible] = -1.0  # Interaction term σ^z_i σ^z_{i+1}
    H[i, i] = -0.5  # Transverse field term -h σ^x_i
rbm.time_evolution(H)